# PW26 Project 4 executed verification notebook
Target journal: Telecommunications Policy. This notebook replays the frozen 300-repository sample offline. Run it from the extracted bundle directory. The deployment review is exploratory and the full-sample results support a descriptive analysis of repository metadata and provider visibility.


In [1]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, quote, urlsplit, urljoin
from urllib.request import Request, urlopen
import csv
import json
import os
import random
import re
import time
import xml.etree.ElementTree as ET

MODE = "live"
REPLAY = True
N = 150
SEED = 42
ROOT = Path("reproducibility")
RAW, OUT = ROOT / "raw", ROOT / "outputs"
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
assert MODE in {"fixture", "live"} and N >= 4
HF = "https://huggingface.co"

def now():
    return datetime.now(timezone.utc).isoformat()

def save_json(path, value):
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def table(path, columns, rows):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

def request(url, method="GET"):
    headers = {"User-Agent": "PW26-deployment-study/1.0"}
    for attempt in range(3):
        try:
            with urlopen(Request(url, method=method, headers=headers), timeout=25) as response:
                data = response.read() if method == "GET" else b""
                return response.status, response.headers, data
        except HTTPError as e:
            if e.code not in {429, 500, 502, 503, 504}:
                return e.code, e.headers, b""
            retry = e.headers.get("Retry-After", "")
            delay = min(int(retry), 60) if retry.isdigit() else 2 ** attempt
        except (URLError, TimeoutError, OSError):
            delay = 2 ** attempt
        if attempt < 2:
            time.sleep(delay)
    return 0, {}, b""

def decode_json(data, expected):
    try:
        value = json.loads(data)
    except (UnicodeError, ValueError):
        return None
    return value if isinstance(value, expected) else None

def hhi(counter):
    total = sum(counter.values())
    return sum((value / total) ** 2 for value in counter.values()) if total else None

def pct(num, denom):
    return round(100 * num / denom, 4) if denom else None

## 1 · Freeze the frame and select two cohorts

The primary cohort contains 150 repositories selected from four download-rank quartiles. The sensitivity cohort contains the 150 most downloaded. Overlap is allowed and reported. Quartile ranks break download ties by repository ID. Stratified allocation uses equal quotas with reproducible simple random sampling within each quartile. The frame is the public catalogue returned under the exact `text-generation` filter during capture. We sample before checking weight access so that gating, missing artifacts and failed observations are visible. All RQ2 rates are conditional on observed downloadable artifacts and their relevant evidence. Downloads represent platform activity.

In [2]:
frame_path = RAW / "catalogue.json"
if MODE == "fixture":
    frame = {
        "started_utc": None, "finished_utc": None,
        "source": "synthetic", "models": [
            {"id": f"fixture-org-{i % 12}/model-{i:03d}", "downloads": (i * 7) % 371}
            for i in range(max(240, N))
        ]
    }
    save_json(frame_path, frame)
elif frame_path.exists():
    frame = load_json(frame_path)
elif REPLAY:
    raise FileNotFoundError("Replay requires raw/catalogue.json from a live capture.")
else:
    url = HF + "/api/models?" + urlencode(
        {"pipeline_tag": "text-generation", "sort": "downloads", "direction": -1,
         "limit": 1000, "full": "false"}
    )
    frame = {"started_utc": now(), "source": url, "models": []}
    seen_pages = set()
    while url:
        if url in seen_pages or urlsplit(url).netloc != "huggingface.co":
            raise RuntimeError("Invalid catalogue pagination. No partial frame was committed.")
        seen_pages.add(url)
        status, headers, data = request(url)
        if status != 200:
            raise RuntimeError(f"Catalogue fetch failed ({status}). Rerun with public Hub access.")
        page = decode_json(data, list)
        if page is None or any(not isinstance(row, dict) for row in page):
            raise ValueError("Expected a list of catalogue records.")
        frame["models"].extend({"id": row.get("id") or row.get("modelId"),
                                "downloads": row.get("downloads")} for row in page)
        link = re.search(r'<([^>]+)>\s*;\s*rel="?next"?', headers.get("Link", ""))
        url = urljoin(url, link.group(1)) if link else None
    frame["finished_utc"] = now()
    frame["pages"] = len(seen_pages)
    save_json(frame_path, frame)

catalogue = {}
for row in frame["models"]:
    repo, downloads = row["id"], row["downloads"]
    if isinstance(repo, str) and repo and type(downloads) is int and downloads >= 0:
        catalogue.setdefault(repo, downloads)
ranked = sorted(catalogue, key=lambda repo: (-catalogue[repo], repo.casefold(), repo))
if len(ranked) < N:
    raise RuntimeError(f"Need {N} eligible repositories with recorded downloads; found {len(ranked)}.")
strata = [ranked[i * len(ranked) // 4:(i + 1) * len(ranked) // 4] for i in range(4)]
rng = random.Random(SEED)
cohorts = {
    "stratified": sorted(repo for i, stratum in enumerate(strata)
                         for repo in rng.sample(stratum, N // 4 + int(i < N % 4))),
    "top_downloads": ranked[:N],
}
models = sorted(set().union(*map(set, cohorts.values())))
quartile = {repo: i + 1 for i, group in enumerate(strata) for repo in group}
table(OUT / "sample.csv", ["cohort", "repo", "downloads", "download_quartile"],
      [{"cohort": cohort, "repo": repo, "downloads": catalogue[repo],
        "download_quartile": quartile[repo]} for cohort, repos in cohorts.items() for repo in repos])
print(MODE, "| eligible:", len(ranked), "| dropped or duplicate rows:",
      len(frame["models"]) - len(ranked), "| overlap:", 2 * N - len(models))

live | eligible: 436004 | dropped or duplicate rows: 0 | overlap: 0


## 2 · Archive revision-level observations

Model-card lineage is a declaration. The weight probe follows a revision-specific anonymous HEAD request for one preferred candidate artifact. A positive result establishes reachability of that artifact only. For sharded weights or adapters, complete inference weights require additional verification in `review_models.csv`. API provider listings are leads for reviewer verification. They are retained in the raw JSON.

In [3]:
def fetch_model(repo):
    key = sha256(repo.encode()).hexdigest()[:20]
    path = RAW / (key + ".json")
    if path.exists():
        return load_json(path)
    escaped = quote(repo, safe="/")
    endpoint = HF + "/api/models/" + escaped
    status, _, payload = request(endpoint)
    result = {"repo": repo, "observed_utc": now(), "metadata_status": status}
    if status != 200:
        return result
    info = decode_json(payload, dict)
    rev = (info or {}).get("sha")
    if not isinstance(rev, str) or not re.fullmatch(r"[0-9a-f]{40}", rev):
        result["metadata_issue"] = "Invalid response or missing commit SHA"
        return result
    result.update(revision=rev, metadata=info)
    base = HF + "/" + escaped + "/resolve/" + quote(rev, safe="")
    status, _, card = request(base + "/README.md")
    result["card_status"] = status
    if status == 200:
        result["card"] = card.decode("utf-8", errors="replace")
    pstatus, _, provider = request(endpoint + "?expand=inferenceProviderMapping")
    result["provider_status"] = pstatus
    if pstatus == 200:
        result["provider_listing"] = decode_json(provider, dict)
        if result["provider_listing"] is None:
            result["provider_status"] = 0  # allow a fresh attempt on the next capture
    siblings = [item["rfilename"] for item in (info.get("siblings") or [])
                if isinstance(item, dict) and isinstance(item.get("rfilename"), str)]
    candidates = sorted(name for name in siblings if re.search(
        r"(?i)(\.safetensors|\.gguf|pytorch_model[^/]*\.bin)$", name))
    if candidates:
        chosen = min(candidates, key=lambda name: (
            "adapter" in name.lower(), bool(re.search(r"\d+-of-\d+", name)), name))
        status, _, _ = request(base + "/" + quote(chosen, safe="/"), "HEAD")
        result.update(weight_probe_path=chosen, weight_probe_status=status)
    if all(result.get(k, 200) != 0 for k in ("weight_probe_status", "provider_status", "card_status")):
        save_json(path, result)
    return result

def fixture_model(repo):
    i = int(repo.rsplit("-", 1)[1])
    parent = f"fixture-base/family-{i % 15}"
    tags = [f"base_model:finetune:{parent}", f"base_model:{parent}", "license:apache-2.0"]
    return {"repo": repo, "revision": f"test{i:04d}", "metadata_status": 200,
            "metadata": {"tags": tags}, "weight_probe_status": 200 if i % 5 else 403}

observations_path = RAW / "run_model_observations.json"
if MODE == "live" and REPLAY:
    records = load_json(observations_path)
    if set(records) != set(models) or any(rec.get("repo") != repo for repo, rec in records.items()):
        raise ValueError("Archived observations do not match the selected catalogue and sample.")
else:
    records = {repo: (fixture_model(repo) if MODE == "fixture" else fetch_model(repo)) for repo in models}
    save_json(observations_path, records)

def declared_parents(record):
    found = set()
    for tag in (record.get("metadata", {}).get("tags") or []):
        if isinstance(tag, str) and tag.startswith("base_model:"):
            repo = tag.rsplit(":", 1)[-1].strip()
            if re.fullmatch(r"[\w.-]+(?:/[\w.-]+)?", repo) and repo.lower() != "unknown":
                found.add(repo.casefold())
    return found

## 3 · Review official documentation

Fill in the live CSVs under the run directory. Leave unavailable values blank or use `unknown` in categorical fields. Every affirmative finding requires the official URL, an observation date, and a short evidence note. Check the evidence against the recorded commit and retain a permitted excerpt or archived reference in the note. Record reviewer identity and independently double-code a subset before publication.

- **review_models.csv** · Use stable canonical IDs for developer and corporate parent. A namespace is a lead. Where a developer is verified to operate independently, use its developer ID as the parent and cite the evidence. For weights use `downloadable`, `gated`, `unavailable` or `unknown`. Confirm complete usable weights (including all necessary shards or external base weights). For licences use `permissive`, `restrictive`, or `unknown`. Code restrictive when official terms impose field-of-use, noncommercial, or additional deployment permission conditions. For other legal conditions, describe the terms and leave ambiguous cases unknown. Use `min_vram_gib` only for documented minimum aggregate GPU memory at batch 1, 2,048-token context, with the lowest documented compatible precision; include runtime, precision and configuration in the note. Leave unknown if the source cannot support that comparison. The threshold of more than 24 GiB is an analytical choice; 16 and 48 GiB are sensitivity thresholds.
- **review_routes.csv** · One row per model, route and runtime. Use `independent_host` for a verified operator distinct from the verified developer and corporate parent, `developer_host` for an affiliated operator, and `self_host` for a documented local recipe. Use the operator's canonical ID as `operator`; use `local` for self-hosting. Verify access, exact model variant, and usable configuration. A provider listing alone does not establish independence, service availability, or a shared backend.
- **review_lineage.csv** · Use lower-case model IDs. Record each verified direct parent as `parent`, or mark a verified terminal model `root` with an empty parent. Include rows for intermediate models and roots outside the sample. Use `parent_set_complete=yes` only once all direct parents are verified, including merges. Use official model documentation for every edge. Report unresolved paths explicitly.

In [4]:
model_fields = ("repo revision reviewer observed_date developer developer_url corporate_parent owner_url "
                "weights weights_url licence licence_url min_vram_gib hardware_url note").split()
route_fields = "repo revision reviewer observed_date route operator runtime url note".split()
lineage_fields = "child relationship parent parent_set_complete reviewer observed_date url note".split()

def review_rows(name, fields, defaults=()):
    path = ROOT / name
    if not path.exists():
        if REPLAY:
            raise FileNotFoundError(f"Replay requires {path.name} from the reviewed archive.")
        table(path, fields, defaults)
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != fields:
            raise ValueError(f"{name}: preserve the generated column names and order.")
        rows = []
        for row in reader:
            if None in row:
                raise ValueError(f"{name} line {reader.line_num}: extra columns. Quote notes containing commas.")
            if any(row.values()):
                rows.append({k: (v or "").strip() for k, v in row.items()})
    return rows

def evidence(row, key):
    try:
        url = urlsplit(row.get(key, ""))
        date = datetime.strptime(row.get("observed_date", ""), "%Y-%m-%d").date()
    except ValueError:
        return False
    return bool(url.scheme == "https" and url.hostname and date <= datetime.now(timezone.utc).date()
                and row.get("reviewer") and row.get("note"))

def make_fixture_reviews():
    model_rows, route_rows, lineage_rows = [], [], {}
    meta = dict(reviewer="synthetic", observed_date="2025-01-01", note="Synthetic test case.")
    url = "https://example.invalid/synthetic"
    for repo, record in records.items():
        i = int(repo.rsplit("-", 1)[1])
        key = dict(repo=repo, revision=record["revision"])
        model_rows.append(dict(key, **meta, developer=f"dev-{i % 12}", developer_url=url,
            corporate_parent=f"parent-{i % 4}", owner_url=url,
            weights="gated" if i % 5 == 0 else "downloadable", weights_url=url,
            licence="restrictive" if i % 3 == 0 else "permissive", licence_url=url,
            min_vram_gib=str(12 * (1 + i % 4)), hardware_url=url))
        if i % 5:
            for j in range(1 + i % 2):
                route_rows.append(dict(key, **meta, route="independent_host",
                    operator=f"host-{(i + j) % 5}", runtime=f"runtime-{j}", url=url))
            if i % 3 == 0:
                route_rows.append(dict(key, **meta, route="self_host", operator="local",
                                       runtime="runtime-local", url=url))
        for parent in declared_parents(record):
            lineage_rows[repo] = dict(child=repo, parent=parent, relationship="parent",
                                      parent_set_complete="yes", **meta, url=url)
            lineage_rows[parent] = dict(child=parent, parent="", relationship="root",
                                        parent_set_complete="yes", **meta, url=url)
    return model_rows, route_rows, list(lineage_rows.values())

if MODE == "fixture":
    reviews, routes, lineage = make_fixture_reviews()
else:
    reviews = review_rows("review_models.csv", model_fields,
                          [{"repo": repo, "revision": rec.get("revision", "")}
                           for repo, rec in records.items()])
    existing = {row["repo"] for row in reviews}
    pending = []
    for row in reviews:
        revision = records.get(row["repo"], {}).get("revision", "")
        if revision and not row["revision"]:
            if any(row[field] for field in model_fields if field not in {"repo", "revision"}):
                continue  # never silently bind previously coded evidence to a newly fetched revision
            row["revision"] = revision
    pending.extend({"repo": repo, "revision": rec.get("revision", "")}
                   for repo, rec in records.items() if repo not in existing)
    if not REPLAY:
        table(ROOT / "review_models.csv", model_fields, [*reviews, *pending])
    routes = review_rows("review_routes.csv", route_fields)
    lineage = review_rows("review_lineage.csv", lineage_fields)

audit = []
def flag(entity, problem):
    audit.append({"entity": entity, "problem": problem})

def matches(row):
    repo = row["repo"]
    return repo in records and bool(row["revision"]) and row["revision"] == records[repo].get("revision")

def model_reviewed(row):
    return any(row.get(field) for field in model_fields if field not in {"repo", "revision"})

reviewed = {}
duplicate_reviews = {repo for repo, count in Counter(
    row["repo"] for row in reviews if model_reviewed(row)).items() if count > 1}
for row in reviews:
    repo = row["repo"]
    if not model_reviewed(row):
        continue
    if not matches(row) or repo in duplicate_reviews:
        flag(repo, "Model review has missing or mismatched revision, or duplicate rows.")
        continue
    item = {}
    for field, url in (("developer", "developer_url"), ("corporate_parent", "owner_url"),
                       ("weights", "weights_url"), ("licence", "licence_url"),
                       ("min_vram_gib", "hardware_url")):
        value = row.get(field, "")
        if not value or value == "unknown":
            continue
        if not evidence(row, url):
            flag(repo, f"Missing source, reviewer, date or note for {field}.")
            continue
        if field == "min_vram_gib":
            try:
                value = float(value)
                if not 0 < value < float("inf"):
                    raise ValueError
            except ValueError:
                flag(repo, "Invalid minimum VRAM.")
                continue
        elif field == "weights" and value not in {"downloadable", "gated", "unavailable"}:
            flag(repo, "Invalid weights category.")
            continue
        elif field == "licence" and value not in {"permissive", "restrictive"}:
            flag(repo, "Invalid licence category.")
            continue
        item[field] = value
        item[url] = row[url]
    reviewed[repo] = item

owners_by_developer = defaultdict(set)
for item in reviewed.values():
    if item.get("developer") and item.get("corporate_parent"):
        owners_by_developer[item["developer"]].add(item["corporate_parent"])
for repo, item in reviewed.items():
    if len(owners_by_developer.get(item.get("developer"), set())) > 1:
        flag(repo, "Conflicting corporate parents for canonical developer; reconcile evidence.")
        item.pop("corporate_parent", None)
        item.pop("owner_url", None)

valid_routes = defaultdict(dict)
for row in routes:
    repo, route, operator, runtime = (row[k] for k in ("repo", "route", "operator", "runtime"))
    owner = reviewed.get(repo, {})
    valid_independent = (owner.get("developer") and owner.get("corporate_parent")
                         and operator not in {owner["developer"], owner["corporate_parent"], "local"})
    valid_affiliated = operator in {owner.get("developer"), owner.get("corporate_parent")}
    valid_route = {"independent_host": valid_independent, "developer_host": valid_affiliated,
                   "self_host": operator == "local"}.get(route, False)
    if not (matches(row) and evidence(row, "url") and operator and runtime and valid_route):
        flag(repo, "Route lacks revision, documented configuration or verified operator relationship.")
        continue
    valid_routes[repo][(route, operator, runtime)] = row["url"]

parents, roots, completeness = defaultdict(set), set(), defaultdict(list)
lineage_sources = {}
for row in lineage:
    child, parent = row["child"].casefold(), row["parent"].casefold()
    relationship = row["relationship"]
    if (not child or not evidence(row, "url") or row["parent_set_complete"] not in {"yes", "no"}
            or (relationship, bool(parent)) not in {("root", False), ("parent", True)}):
        flag(child, "Invalid lineage review.")
        continue
    completeness[child].append(row["parent_set_complete"] == "yes")
    if relationship == "root":
        roots.add(child)
    else:
        parents[child].add(parent)
    lineage_sources[child, parent] = row["url"]

def resolve_roots(node, path=frozenset()):
    if node in path or (node in roots and parents.get(node)):
        return set(), False
    if node in roots:
        return {node}, all(completeness[node])
    if not parents.get(node):
        return set(), False
    combined = set()
    complete = all(completeness[node])
    for parent in parents[node]:
        resolved, status = resolve_roots(parent, path | {node})
        combined.update(resolved)
        complete = complete and status
    return combined, complete

ancestry = {repo: resolve_roots(repo.casefold()) for repo in models}
for repo, (terminal, complete) in ancestry.items():
    missing = declared_parents(records[repo]) - parents.get(repo.casefold(), set())
    if complete and missing:
        flag(repo, "Reviewed lineage omits declared parent(s); resolve before assigning root signature.")
        ancestry[repo] = (terminal, False)

## 4 · Build the dependency graph and answer RQ1–RQ4

Graph edges distinguish verified corporate relationships, documented routes, and declared lineage. Hosting concentration uses fractional incidence: each repository with documented managed routes contributes one unit divided across distinct operators. This is a measure of documented options. For RQ2 report the verified downloadable denominator, complete-case rates, and missing-data bounds. For RQ3 compare distinct groups on matched denominators. For RQ4 remove each operator from the documented route set and count the models that lose every recorded route; local routes count as fallbacks. A loss in this scenario does not establish actual service dependence or exclusivity.

In [5]:
nodes, edges = {}, set()
def edge(source_type, source, target_type, target, kind, url):
    source_id, target_id = f"{source_type}:{source}", f"{target_type}:{target}"
    nodes[source_id] = source_type, source
    nodes[target_id] = target_type, target
    edges.add((source_id, target_id, kind, url))

for repo in models:
    record, item = records[repo], reviewed.get(repo, {})
    node = "model:" + repo.casefold()
    nodes[node] = "model", repo.casefold()
    model_url = (HF + "/" + repo + "/tree/" + record["revision"]
                 if record.get("revision") else HF + "/" + repo)
    for parent in declared_parents(record):
        edge("model", repo.casefold(), "model", parent, "declares_direct_base", model_url)
    if item.get("developer"):
        edge("model", repo.casefold(), "developer", item["developer"],
             "developed_by", item["developer_url"])
    if item.get("developer") and item.get("corporate_parent"):
        edge("developer", item["developer"], "corporate_parent", item["corporate_parent"],
             "verified_parent", item["owner_url"])
    for (kind, operator, runtime), url in valid_routes[repo].items():
        edge("model", repo.casefold(), "runtime", runtime, "documented_runtime", url)
        edge("model", repo.casefold(), "host", operator, kind, url)
for (child, parent), url in lineage_sources.items():
    if parent:
        edge("model", child, "model", parent, "verified_direct_base", url)

table(OUT / "graph_nodes.csv", ["id", "kind", "label"],
      [{"id": key, "kind": kind, "label": label} for key, (kind, label) in sorted(nodes.items())])
table(OUT / "graph_edges.csv", ["source", "target", "relation", "evidence_url"],
      [dict(zip(("source", "target", "relation", "evidence_url"), item)) for item in sorted(edges)])
graphml = ET.Element("graphml", xmlns="http://graphml.graphdrawing.org/xmlns")
for key, scope in (("kind", "node"), ("label", "node"), ("relation", "edge"), ("evidence_url", "edge")):
    ET.SubElement(graphml, "key", {"id": key, "for": scope, "attr.name": key, "attr.type": "string"})
graph = ET.SubElement(graphml, "graph", edgedefault="directed")
graph_ids = {}
for i, (key, (kind, label)) in enumerate(sorted(nodes.items())):
    graph_ids[key] = f"n{i}"
    node = ET.SubElement(graph, "node", id=graph_ids[key])
    ET.SubElement(node, "data", key="kind").text = kind
    ET.SubElement(node, "data", key="label").text = label
for i, (source, target, relation, url) in enumerate(sorted(edges)):
    element = ET.SubElement(graph, "edge", id=f"e{i}", source=graph_ids[source], target=graph_ids[target])
    ET.SubElement(element, "data", key="relation").text = relation
    ET.SubElement(element, "data", key="evidence_url").text = url
ET.ElementTree(graphml).write(OUT / "dependency_graph.graphml", encoding="utf-8", xml_declaration=True)

def options(repo, kinds):
    return {operator for kind, operator, _ in valid_routes[repo] if kind in kinds}

MANAGED = {"independent_host", "developer_host"}
ALL_ROUTES = MANAGED | {"self_host"}
concentration, restrictions, diversity, portability, summaries, detail = [], [], [], [], [], []
for cohort, repos in cohorts.items():
    info = {repo: reviewed.get(repo, {}) for repo in repos}
    for name, per_repo in {
        "verified_developer": {repo: {item["developer"]} if "developer" in item else set()
                               for repo, item in info.items()},
        "verified_corporate_parent": {repo: {item["corporate_parent"]} if "corporate_parent" in item else set()
                                     for repo, item in info.items()},
        "documented_managed_host": {repo: options(repo, MANAGED) for repo in repos},
        "documented_independent_host": {repo: options(repo, {"independent_host"}) for repo in repos},
        "declared_direct_base": {repo: declared_parents(records[repo]) for repo in repos},
    }.items():
        counts = Counter()
        for labels in per_repo.values():
            for label in labels:
                counts[label] += 1 / len(labels)
        covered = sum(bool(labels) for labels in per_repo.values())
        index = hhi(counts)
        concentration.append(dict(cohort=cohort, dimension=name, sample_n=len(repos), covered_n=covered,
            unique_entities=len(counts), hhi=index, effective_entities=1/index if index else None,
            top_entity=counts.most_common(1)[0][0] if counts else "",
            top_entity_pct=pct(max(counts.values()), covered) if counts else None))

    downloadable = [repo for repo in repos if info[repo].get("weights") == "downloadable"]
    for condition in ("restrictive_licence", "vram_gt_16", "vram_gt_24", "vram_gt_48"):
        field = "licence" if condition == "restrictive_licence" else "min_vram_gib"
        verified = [info[repo][field] for repo in downloadable if field in info[repo]]
        if field == "licence":
            positive = sum(value == "restrictive" for value in verified)
        else:
            threshold = int(condition.rsplit("_", 1)[1])
            positive = sum(value > threshold for value in verified)
        missing = len(downloadable) - len(verified)
        restrictions.append(dict(cohort=cohort, condition=condition, sample_n=len(repos),
            weights_status_known_n=sum("weights" in item for item in info.values()),
            verified_downloadable_n=len(downloadable), known_condition_n=len(verified),
            positive_n=positive, complete_case_pct=pct(positive, len(verified)),
            lower_pct=pct(positive, len(downloadable)),
            upper_pct=pct(positive + missing, len(downloadable))))

    direct = {repo: tuple(sorted(declared_parents(records[repo]))) for repo in repos}
    root_signatures = {repo: tuple(sorted(ancestry[repo][0])) for repo in repos if ancestry[repo][1]}
    for name, signatures in (
        ("declared_direct_parent_set", {repo: sig for repo, sig in direct.items() if sig}),
        ("verified_terminal_root_set", root_signatures),
        ("verified_corporate_parent", {repo: item["corporate_parent"] for repo, item in info.items()
                                        if "corporate_parent" in item}),
        ("joint_root_set_and_parent", {repo: (sig, info[repo]["corporate_parent"])
                                      for repo, sig in root_signatures.items() if "corporate_parent" in info[repo]}),
    ):
        richness = len(set(signatures.values()))
        diversity.append(dict(cohort=cohort, grouping=name, sample_n=len(repos), matched_repo_n=len(signatures),
            unique_groups=richness, reduction_pct=pct(len(signatures)-richness, len(signatures))))

    all_options = {repo: options(repo, ALL_ROUTES) for repo in repos}
    independent = {repo: options(repo, {"independent_host"}) for repo in repos}
    documented = sum(bool(choices) for choices in all_options.values())
    summaries.append(dict(cohort=cohort, sample_n=len(repos), documented_route_n=documented,
        independent_host_n=sum(bool(choices) for choices in independent.values()),
        two_independent_hosts_n=sum(len(choices) >= 2 for choices in independent.values()),
        self_host_n=sum("local" in choices for choices in all_options.values()),
        multiple_documented_runtimes_n=sum(len({rt for _, _, rt in valid_routes[repo]}) >= 2 for repo in repos)))
    operators = sorted(set().union(*all_options.values()) - {"local"})
    for operator in operators:
        exposed = [repo for repo, choices in all_options.items() if operator in choices]
        loses = sum(all_options[repo] == {operator} for repo in exposed)
        portability.append(dict(cohort=cohort, removed_operator=operator, exposed_n=len(exposed),
            loses_all_documented_routes_n=loses, retains_documented_alternative_n=len(exposed)-loses,
            pct_of_models_with_documented_route=pct(loses, documented)))
    for repo in repos:
        item, record = info[repo], records[repo]
        detail.append(dict(cohort=cohort, repo=repo, revision=record.get("revision", ""),
            developer=item.get("developer", ""), corporate_parent=item.get("corporate_parent", ""),
            reviewed_weights=item.get("weights", "unknown"), artifact_status=record.get("weight_probe_status"),
            artifact=record.get("weight_probe_path", ""), reviewed_licence=item.get("licence", "unknown"),
            min_vram_gib=item.get("min_vram_gib"), complete_lineage=ancestry[repo][1],
            declared_direct_bases="|".join(direct[repo]), verified_roots="|".join(sorted(ancestry[repo][0])),
            independent_hosts="|".join(sorted(independent[repo])),
            documented_runtimes="|".join(sorted({rt for _, _, rt in valid_routes[repo]}))))

for filename, rows in (("rq1_concentration", concentration), ("rq2_constraints", restrictions),
                       ("rq3_diversity", diversity), ("rq4_route_coverage", summaries),
                       ("model_evidence", detail)):
    table(OUT / (filename + ".csv"), list(rows[0]), rows)
table(OUT / "rq4_operator_removal.csv", ["cohort", "removed_operator", "exposed_n",
      "loses_all_documented_routes_n", "retains_documented_alternative_n",
      "pct_of_models_with_documented_route"], portability)
table(OUT / "review_issues.csv", ["entity", "problem"], audit)

## 5 · Verification and publication decision

Before using RQ4 in a manuscript, select examples from the verified route table. Describe model revision, runtime, hardware, operator, and evidence for each alternative. Cross-runtime recipes support documented portability; actual interoperability claims require deployment validation. For single documented routes, describe what evidence would establish a second route and tie the proposal to the relevant observed barrier. Route and licence coverage are reported alongside all estimates.

In [6]:
assert all(len(repos) == len(set(repos)) == N for repos in cohorts.values())
assert set(cohorts["stratified"]).issubset(catalogue)
assert cohorts["top_downloads"] == ranked[:N]
assert sum(len(repos) for repos in cohorts.values()) == len(detail)
assert all(source in nodes and target in nodes for source, target, _, _ in edges)
assert all(row["hhi"] is None or 0 < row["hhi"] <= 1.0000001 for row in concentration)
assert all(row["positive_n"] <= row["known_condition_n"] <= row["verified_downloadable_n"]
           for row in restrictions)
assert all(row["unique_groups"] <= row["matched_repo_n"] <= N for row in diversity)
assert all(row["loses_all_documented_routes_n"] + row["retains_documented_alternative_n"]
           == row["exposed_n"] for row in portability)
if MODE == "fixture":
    assert not audit
    assert all(row["covered_n"] > 0 for row in concentration)
    assert all(row["known_condition_n"] > 0 for row in restrictions)
    assert all(row["matched_repo_n"] == N for row in diversity)
    assert portability and any(row["loses_all_documented_routes_n"] > 0 for row in portability)
    assert resolve_roots("fictional/unreviewed") == (set(), False)
    parents["fictional/cycle"] = {"fictional/cycle"}
    assert resolve_roots("fictional/cycle") == (set(), False)
    del parents["fictional/cycle"]

verdict = {
    "mode": MODE,
    "technical_workflow": "Analysis workflow ran and passed consistency checks.",
    "empirical_status": ("Synthetic test only." if MODE == "fixture" else
                         "Inspect coverage and audit outputs; independent evidence review is required."),
    "publication_ready": False,
    "publication_status": "A real, independently reviewed analysis and a complete manuscript are required.",
    "records_with_revision": sum(bool(record.get("revision")) for record in records.values()),
    "model_card_http_status_counts": dict(Counter(str(rec.get("card_status", "unobserved")) for rec in records.values())),
    "developer_evidence_n": sum("developer" in item for item in reviewed.values()),
    "ownership_evidence_n": sum("corporate_parent" in item for item in reviewed.values()),
    "weight_evidence_n": sum("weights" in item for item in reviewed.values()),
    "documented_route_n": sum(bool(valid_routes[repo]) for repo in models),
    "resolved_lineage_n": sum(complete for _, complete in ancestry.values()),
    "sample_unique_repositories": len(models),
    "cohort_overlap_n": 2 * N - len(models),
    "cohort_n_each": N, "seed": SEED,
    "frame_eligible_n": len(ranked), "catalogue_capture": {
        key: value for key, value in frame.items() if key != "models"
    },
    "reviews_with_issues": len(audit),
    "inputs_sha256": {
        str(p.relative_to(ROOT)): sha256(p.read_bytes()).hexdigest()
        for p in sorted([*RAW.glob("*.json"), *ROOT.glob("review_*.csv")])
    },
}
save_json(OUT / "run_manifest.json", verdict)
print(json.dumps({k: v for k, v in verdict.items() if k != "inputs_sha256"}, indent=2))
print("Results:", OUT.resolve())

{
  "mode": "live",
  "technical_workflow": "Analysis workflow ran and passed consistency checks.",
  "empirical_status": "Inspect coverage and audit outputs; independent evidence review is required.",
  "publication_ready": false,
  "publication_status": "A real, independently reviewed analysis and a complete manuscript are required.",
  "records_with_revision": 300,
  "model_card_http_status_counts": {
    "200": 286,
    "404": 14
  },
  "developer_evidence_n": 14,
  "ownership_evidence_n": 3,
  "weight_evidence_n": 0,
  "documented_route_n": 3,
  "resolved_lineage_n": 1,
  "sample_unique_repositories": 300,
  "cohort_overlap_n": 0,
  "cohort_n_each": 150,
  "seed": 42,
  "frame_eligible_n": 436004,
  "catalogue_capture": {
    "started_utc": "2026-11-10T16:33:29.269764+00:00",
    "source": "https://huggingface.co/api/models?pipeline_tag=text-generation&sort=downloads&direction=-1&limit=1000&full=false",
    "finished_utc": "2026-11-10T16:36:48.790221+00:00",
    "pages": 437
  },


## Reproduce the manuscript tables
These separate platform tags and provider listings from the reviewed deployment evidence.

In [7]:
import runpy
_ = runpy.run_path("analyze_snapshot.py", run_name="__main__")


[
  {
    "cohort": "stratified",
    "n": 150,
    "card_ok": 137,
    "license_tag_n": 86,
    "declared_base_n": 75,
    "distinct_declared_bases": 80,
    "direct_base_hhi": 0.014562962962962965,
    "namespaces": 145,
    "namespace_hhi": 0.0072000000000000015,
    "gated_n": 2,
    "weight_http_2xx_n": 140,
    "weight_probe_observed_n": 142,
    "provider_mapping_n": 150,
    "listed_live_provider_n": 3,
    "multiple_listed_live_provider_n": 0,
    "distinct_listed_live_providers": 1,
    "listed_provider_hhi": 1.0,
    "reviewed_developer_n": 2,
    "reviewed_parent_n": 0,
    "reviewed_licence_n": 2
  },
  {
    "cohort": "top_downloads",
    "n": 150,
    "card_ok": 149,
    "license_tag_n": 145,
    "declared_base_n": 84,
    "distinct_declared_bases": 68,
    "direct_base_hhi": 0.018140589569160995,
    "namespaces": 55,
    "namespace_hhi": 0.08986666666666668,
    "gated_n": 5,
    "weight_http_2xx_n": 145,
    "weight_probe_observed_n": 150,
    "provider_mapping_n": 14